# 04 — Spatial Probability Map

For a chosen site and query date, runs inference on every `loc` tile and
plots each tile's footprint colored by P(bleached).

Each query tile is paired with the healthy reference tile that shares the
same `loc` ID (same spatial footprint, different date). Tile coordinates
are read from the GeoTIFF geotransform so the output is spatially accurate.

**Data layout expected:**
```
<DATA_ROOT>/<SITE>/bleached/tiled_360m/<QUERY_DATE>/loc*.tif  ← query tiles
<DATA_ROOT>/<SITE>/healthy/tiled_360m/<REF_DATE>/loc*.tif    ← reference tiles
```

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
CKPT_PATH   = "/pscratch/sd/k/kevinval/coraltest/ct_classifier/model_states/best.pt"
PROJECT_DIR = "/pscratch/sd/k/kevinval/coraltest/ct_classifier"
DATA_ROOT   = "/pscratch/sd/k/kevinval/coraltest/ct_classifier/planet_superdove_landmasked"
FIGURES_DIR = "figures"

SITE        = "cheeca_flkeys"   # site to map
QUERY_DATE  = "20230806"        # bleaching event date (YYYYMMDD)
REF_DATE    = "20210801"        # healthy reference date (YYYYMMDD)

In [ ]:
import os, sys, glob
import yaml
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio
from rasterio.transform import array_bounds
import torch

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from ct_classifier.model import CustomResNet
from utils.viz import to_display_rgb, percentile_stretch

os.makedirs(FIGURES_DIR, exist_ok=True)

cfg_path = os.path.join(os.path.dirname(CKPT_PATH), "config.yaml")
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"
norm_factor = float(cfg["normalization_factor"])

model = CustomResNet(cfg["num_classes"], cfg["layers"])
state = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(state["model"])
model.to(device).eval()
print("Model loaded. Device:", device)

In [ ]:
# ── Collect query and reference tiles ─────────────────────────────────────────
query_dir = os.path.join(DATA_ROOT, SITE, "bleached", "tiled_360m", QUERY_DATE)
ref_dir   = os.path.join(DATA_ROOT, SITE, "healthy",  "tiled_360m", REF_DATE)

query_tiles = sorted(glob.glob(os.path.join(query_dir, "loc*.tif")))
ref_tiles   = sorted(glob.glob(os.path.join(ref_dir,   "loc*.tif")))

assert query_tiles, f"No query tiles found in {query_dir}"
assert ref_tiles,   f"No reference tiles found in {ref_dir}"

# Index reference tiles by loc ID so we can match by spatial location
def loc_id(path):
    return os.path.splitext(os.path.basename(path))[0]  # e.g. "loc001"

ref_by_loc = {loc_id(p): p for p in ref_tiles}

# Keep only query tiles that have a matching reference
pairs = [(q, ref_by_loc[loc_id(q)]) for q in query_tiles if loc_id(q) in ref_by_loc]
unmatched = [q for q in query_tiles if loc_id(q) not in ref_by_loc]

print(f"Query tiles : {len(query_tiles)}")
print(f"Ref tiles   : {len(ref_tiles)}")
print(f"Matched pairs: {len(pairs)}")
if unmatched:
    print(f"Unmatched (no ref found): {[loc_id(q) for q in unmatched]}")

In [ ]:
# ── Run inference on all pairs ────────────────────────────────────────────────
def read_tile(path):
    """Returns (arr [8,H,W] float32, valid_mask [H,W] float32, bounds, crs)."""
    with rasterio.open(path) as src:
        arr = src.read().astype(np.float32)
        bounds = src.bounds
        crs    = src.crs
    valid = np.isfinite(arr).all(axis=0).astype(np.float32)
    arr   = np.nan_to_num(arr, nan=0.0)
    return arr, valid, bounds, crs

results = []   # list of {loc, prob, bounds, query_arr}

with torch.no_grad():
    for q_path, r_path in pairs:
        q_arr, q_mask, q_bounds, _ = read_tile(q_path)
        r_arr, r_mask, _,        _ = read_tile(r_path)

        x = np.concatenate([r_arr, q_arr], axis=0) / norm_factor   # [16, H, W]
        m = np.stack([r_mask, q_mask], axis=0)                      # [2, H, W]
        tile = torch.from_numpy(np.concatenate([x, m], axis=0)).unsqueeze(0).to(device)  # [1,18,H,W]

        prob = torch.softmax(model(tile), dim=1)[0, 1].item()

        results.append({
            "loc":       loc_id(q_path),
            "prob":      prob,
            "bounds":    q_bounds,
            "query_arr": q_arr,
        })

probs = [r["prob"] for r in results]
print(f"Inference done on {len(results)} tiles.")
print(f"P(bleached) range: {min(probs):.3f} – {max(probs):.3f}")

In [ ]:
# ── Plot spatial probability map ──────────────────────────────────────────────
# Each tile is drawn as a colored rectangle at its actual geographic location.
cmap = plt.get_cmap("RdYlGn_r")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: RGB mosaic (one RGB chip per tile position)
for r in results:
    b = r["bounds"]
    rgb = percentile_stretch(to_display_rgb(torch.from_numpy(r["query_arr"])))
    axes[0].imshow(
        rgb,
        extent=[b.left, b.right, b.bottom, b.top],
        origin="upper", aspect="auto"
    )

axes[0].set_title(f"{SITE} — {QUERY_DATE} (RGB)")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")

# Right: probability rectangles
for r in results:
    b = r["bounds"]
    color = cmap(r["prob"])
    rect = mpatches.Rectangle(
        (b.left, b.bottom),
        b.right - b.left,
        b.top   - b.bottom,
        linewidth=0.5, edgecolor="white",
        facecolor=color, alpha=0.85
    )
    axes[1].add_patch(rect)
    axes[1].text(
        (b.left + b.right) / 2, (b.bottom + b.top) / 2,
        f"{r['prob']:.2f}",
        ha="center", va="center", fontsize=6, color="black"
    )

# Match axis limits to the tile extents
all_bounds = [r["bounds"] for r in results]
x_min = min(b.left   for b in all_bounds)
x_max = max(b.right  for b in all_bounds)
y_min = min(b.bottom for b in all_bounds)
y_max = max(b.top    for b in all_bounds)
margin = (x_max - x_min) * 0.05

for ax in axes:
    ax.set_xlim(x_min - margin, x_max + margin)
    ax.set_ylim(y_min - margin, y_max + margin)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
sm.set_array([])
plt.colorbar(sm, ax=axes[1], fraction=0.046, pad=0.04, label="P(bleached)")
axes[1].set_title(f"{SITE} — {QUERY_DATE} (bleach probability)")
axes[1].set_xlabel("Longitude")

fig.tight_layout()
fig.savefig(
    os.path.join(FIGURES_DIR, f"spatial_prob_{SITE}_{QUERY_DATE}.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()